In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pandas as pd

# Load dataset
df = pd.read_csv("suicide-watch/suicide_watch_cleaned.csv")

# Clean
df["clean_text"] = df["clean_text"].fillna("").astype(str)
df = df[df["clean_text"].str.strip() != ""]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42
)

# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)  # ✅ IMPORTANT

# Model
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.94      0.94     23092
           1       0.94      0.93      0.94     23311

    accuracy                           0.94     46403
   macro avg       0.94      0.94      0.94     46403
weighted avg       0.94      0.94      0.94     46403



In [4]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

[[21815  1277]
 [ 1724 21587]]


In [5]:
errors = X_test[y_test != y_pred]
print(errors.head(10))

181299    why do i have to work hard doing the bare mini...
143722    for whatever its worthi never stopped loving y...
130909    how do you get over the thought of killing you...
51043     the others agree principle hello fellow suffer...
64583     dar get annoyed when your hear your own heartb...
43154     days until i said i wouldchange of recent even...
224609    im finished i cant go on living anymore im jus...
100641    i dont know if im depressed or a normal teen i...
135268    advice neededasking for a friend whos in schoo...
107404    alcohol is a way to deal with problemsi dont e...
Name: clean_text, dtype: str


In [6]:
samples = [
    "I feel empty and tired of everything",
    "Life is meaningless, I don't see a point",
    "I want to end my life",
    "Today was a great day!"
]

vec = vectorizer.transform(samples)
print(model.predict(vec))

[1 1 1 0]


In [7]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/logistic_model.pkl")
joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")

print("Model and vectorizer saved!")

Model and vectorizer saved!


In [8]:
import joblib

model = joblib.load("models/logistic_model.pkl")
vectorizer = joblib.load("models/tfidf_vectorizer.pkl")

In [9]:
samples = ["I feel hopeless and alone"]

vec = vectorizer.transform(samples)
prediction = model.predict(vec)

print(prediction)

[1]
